# Polynomial Regression: Modeling Non-Linear Relationships

## Overview
Polynomial regression extends linear regression to model non-linear relationships by adding polynomial features. It's still linear regression, but in a transformed feature space.

## Learning Objectives
- Understand when and why to use polynomial regression
- Learn to create polynomial features
- Recognize and address overfitting
- Apply polynomial regression to real-world data
- Compare different polynomial degrees

## Mathematical Foundation

### Simple Polynomial (1 feature)
$$y = \beta_0 + \beta_1 x + \beta_2 x^2 + ... + \beta_d x^d + \epsilon$$

### Multiple Features with Interactions
For degree 2 with features $x_1, x_2$:
$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \beta_3 x_1^2 + \beta_4 x_1 x_2 + \beta_5 x_2^2 + \epsilon$$

### Key Insight
Although the relationship with $y$ is non-linear, it's still **linear in the parameters** $\beta$, so we can use linear regression techniques!

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## 1. Generating Non-Linear Data

In [ ]:
# Generate quadratic data
m = 100
X = 6 * np.random.rand(m, 1) - 3
y = 0.5 * X**2 + X + 2 + np.random.randn(m, 1)

plt.figure(figsize=(10, 6))
plt.scatter(X, y, alpha=0.6, edgecolors='k', s=50)
plt.xlabel('X', fontsize=12)
plt.ylabel('y', fontsize=12)
plt.title('Non-Linear Data (Quadratic Relationship)', fontsize=14)
plt.grid(True, alpha=0.3)
plt.show()

print(f"Data shape: X={X.shape}, y={y.shape}")
print(f"True relationship: y = 0.5*X² + X + 2 + noise")

## 2. Linear Regression (Baseline)

Let's first try simple linear regression to see why it fails on non-linear data.

In [ ]:
# Fit linear regression
lin_reg = LinearRegression()
lin_reg.fit(X, y)
y_pred_linear = lin_reg.predict(X)

# Visualize
X_plot = np.linspace(-3, 3, 100).reshape(-1, 1)
y_plot_linear = lin_reg.predict(X_plot)

plt.figure(figsize=(10, 6))
plt.scatter(X, y, alpha=0.6, edgecolors='k', s=50, label='Data')
plt.plot(X_plot, y_plot_linear, 'r-', linewidth=2, label='Linear Model')
plt.xlabel('X', fontsize=12)
plt.ylabel('y', fontsize=12)
plt.title('Linear Regression on Non-Linear Data', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

mse_linear = mean_squared_error(y, y_pred_linear)
r2_linear = r2_score(y, y_pred_linear)
print(f"Linear Model Performance:")
print(f"  MSE: {mse_linear:.4f}")
print(f"  R² Score: {r2_linear:.4f}")

## 3. Polynomial Features: Manual Implementation

In [ ]:
# Manually create polynomial features
X_poly_manual = np.c_[X, X**2]  # Add X² as a feature

print(f"Original features shape: {X.shape}")
print(f"Polynomial features shape: {X_poly_manual.shape}")
print(f"\nFirst 5 rows:")
print(f"Original X:\n{X[:5]}")
print(f"\nPolynomial features [X, X²]:\n{X_poly_manual[:5]}")

In [ ]:
# Fit polynomial regression
poly_reg_manual = LinearRegression()
poly_reg_manual.fit(X_poly_manual, y)

# Predictions
X_plot_poly = np.c_[X_plot, X_plot**2]
y_plot_poly = poly_reg_manual.predict(X_plot_poly)

plt.figure(figsize=(10, 6))
plt.scatter(X, y, alpha=0.6, edgecolors='k', s=50, label='Data')
plt.plot(X_plot, y_plot_linear, 'r--', linewidth=2, alpha=0.5, label='Linear')
plt.plot(X_plot, y_plot_poly, 'g-', linewidth=2, label='Polynomial (degree 2)')
plt.xlabel('X', fontsize=12)
plt.ylabel('y', fontsize=12)
plt.title('Polynomial Regression vs Linear Regression', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

y_pred_poly = poly_reg_manual.predict(X_poly_manual)
mse_poly = mean_squared_error(y, y_pred_poly)
r2_poly = r2_score(y, y_pred_poly)

print(f"Polynomial Model Performance:")
print(f"  MSE: {mse_poly:.4f}")
print(f"  R² Score: {r2_poly:.4f}")
print(f"\nImprovement over linear model:")
print(f"  MSE reduction: {((mse_linear - mse_poly) / mse_linear * 100):.2f}%")
print(f"  R² increase: {(r2_poly - r2_linear):.4f}")

## 4. Using PolynomialFeatures from Scikit-learn

In [ ]:
# Create polynomial features (degree 2)
poly_features = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly_features.fit_transform(X)

print(f"Feature names: {poly_features.get_feature_names_out()}")
print(f"Shape: {X_poly.shape}")
print(f"\nFirst 5 rows:")
print(X_poly[:5])

## 5. Comparing Different Polynomial Degrees

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

# Test different polynomial degrees
degrees = [1, 2, 3, 4, 10, 20]
results = []

plt.figure(figsize=(15, 10))

for i, degree in enumerate(degrees, 1):
    # Create polynomial features
    poly_features = PolynomialFeatures(degree=degree, include_bias=False)
    X_poly = poly_features.fit_transform(X)
    
    # Fit model
    model = LinearRegression()
    model.fit(X_poly, y)
    
    # Predictions
    y_pred = model.predict(X_poly)
    X_plot_poly = poly_features.transform(X_plot)
    y_plot = model.predict(X_plot_poly)
    
    # Metrics
    mse = mean_squared_error(y, y_pred)
    r2 = r2_score(y, y_pred)
    results.append({'Degree': degree, 'MSE': mse, 'R²': r2})
    
    # Plot
    plt.subplot(2, 3, i)
    plt.scatter(X, y, alpha=0.5, edgecolors='k', s=30)
    plt.plot(X_plot, y_plot, 'r-', linewidth=2)
    plt.xlabel('X', fontsize=10)
    plt.ylabel('y', fontsize=10)
    plt.title(f'Degree {degree} (R²={r2:.3f}, MSE={mse:.3f})', fontsize=11)
    plt.ylim(-5, 12)
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Display results table
results_df = pd.DataFrame(results)
print("\nModel Comparison:")
print(results_df.to_string(index=False))

In [ ]:
# Visualize performance metrics
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MSE vs Degree
axes[0].plot(results_df['Degree'], results_df['MSE'], 'bo-', linewidth=2, markersize=8)
axes[0].set_xlabel('Polynomial Degree', fontsize=12)
axes[0].set_ylabel('Mean Squared Error', fontsize=12)
axes[0].set_title('MSE vs Polynomial Degree', fontsize=14)
axes[0].grid(True, alpha=0.3)
axes[0].set_yscale('log')

# R² vs Degree
axes[1].plot(results_df['Degree'], results_df['R²'], 'ro-', linewidth=2, markersize=8)
axes[1].set_xlabel('Polynomial Degree', fontsize=12)
axes[1].set_ylabel('R² Score', fontsize=12)
axes[1].set_title('R² vs Polynomial Degree', fontsize=14)
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=1.0, color='g', linestyle='--', alpha=0.5, label='Perfect fit')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Overfitting: Training vs Test Performance

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Evaluate different degrees on train and test sets
degrees = range(1, 21)
train_errors = []
test_errors = []

for degree in degrees:
    poly_features = PolynomialFeatures(degree=degree, include_bias=False)
    X_train_poly = poly_features.fit_transform(X_train)
    X_test_poly = poly_features.transform(X_test)
    
    model = LinearRegression()
    model.fit(X_train_poly, y_train)
    
    y_train_pred = model.predict(X_train_poly)
    y_test_pred = model.predict(X_test_poly)
    
    train_errors.append(mean_squared_error(y_train, y_train_pred))
    test_errors.append(mean_squared_error(y_test, y_test_pred))

# Plot learning curves
plt.figure(figsize=(12, 6))
plt.plot(degrees, train_errors, 'b-o', label='Training Error', linewidth=2, markersize=6)
plt.plot(degrees, test_errors, 'r-s', label='Test Error', linewidth=2, markersize=6)
plt.xlabel('Polynomial Degree', fontsize=12)
plt.ylabel('Mean Squared Error', fontsize=12)
plt.title('Training vs Test Error: Detecting Overfitting', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.axvline(x=2, color='g', linestyle='--', alpha=0.5, label='Optimal degree')
plt.show()

optimal_degree = degrees[np.argmin(test_errors)]
print(f"Optimal polynomial degree based on test error: {optimal_degree}")
print(f"Test error at optimal degree: {min(test_errors):.4f}")

## 7. Pipeline: Combining Polynomial Features and Regression

In [ ]:
# Create a pipeline
polynomial_regression = Pipeline([
    ('poly_features', PolynomialFeatures(degree=2, include_bias=False)),
    ('scaler', StandardScaler()),  # Feature scaling
    ('lin_reg', LinearRegression())
])

# Fit the pipeline
polynomial_regression.fit(X_train, y_train)

# Predict
y_train_pred = polynomial_regression.predict(X_train)
y_test_pred = polynomial_regression.predict(X_test)

print("Pipeline Results (Degree 2 with Scaling):")
print(f"Training R²: {r2_score(y_train, y_train_pred):.4f}")
print(f"Test R²: {r2_score(y_test, y_test_pred):.4f}")
print(f"Training RMSE: {np.sqrt(mean_squared_error(y_train, y_train_pred)):.4f}")
print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test, y_test_pred)):.4f}")

## 8. Multiple Features with Polynomial Features

In [ ]:
# Generate data with 2 features
np.random.seed(42)
m = 200
X_multi = np.random.rand(m, 2) * 4 - 2  # 2 features
y_multi = (0.5 * X_multi[:, 0]**2 + 
           X_multi[:, 1] + 
           0.3 * X_multi[:, 0] * X_multi[:, 1] + 
           np.random.randn(m) * 0.5)
y_multi = y_multi.reshape(-1, 1)

print(f"Multi-feature dataset shape: X={X_multi.shape}, y={y_multi.shape}")
print(f"True relationship: y = 0.5*X₁² + X₂ + 0.3*X₁*X₂ + noise")

In [ ]:
# Create polynomial features (degree 2)
poly_multi = PolynomialFeatures(degree=2, include_bias=False)
X_multi_poly = poly_multi.fit_transform(X_multi)

print(f"Original features: {X_multi.shape[1]}")
print(f"Polynomial features (degree 2): {X_multi_poly.shape[1]}")
print(f"\nFeature names: {poly_multi.get_feature_names_out()}")
print(f"\nThese include: original features, squares, and interaction terms")

In [ ]:
# Compare linear vs polynomial regression
X_train, X_test, y_train, y_test = train_test_split(X_multi, y_multi, test_size=0.2, random_state=42)

# Linear model
lin_model = LinearRegression()
lin_model.fit(X_train, y_train)
y_pred_lin = lin_model.predict(X_test)

# Polynomial model
poly_pipeline = Pipeline([
    ('poly', PolynomialFeatures(degree=2)),
    ('linear', LinearRegression())
])
poly_pipeline.fit(X_train, y_train)
y_pred_poly = poly_pipeline.predict(X_test)

print("Comparison on Multi-feature Data:")
print(f"\nLinear Regression:")
print(f"  R² Score: {r2_score(y_test, y_pred_lin):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_lin)):.4f}")
print(f"\nPolynomial Regression (degree 2):")
print(f"  R² Score: {r2_score(y_test, y_pred_poly):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_poly)):.4f}")

## 9. Real-World Example: Temperature vs Ice Cream Sales

A classic non-linear relationship.

In [ ]:
# Generate synthetic ice cream sales data
np.random.seed(42)
temperature = np.random.uniform(10, 40, 100).reshape(-1, 1)  # Temperature in Celsius
# Sales increase with temperature, but plateau at high temps
sales = 100 + 15 * temperature - 0.15 * temperature**2 + np.random.normal(0, 20, (100, 1))

plt.figure(figsize=(10, 6))
plt.scatter(temperature, sales, alpha=0.6, edgecolors='k', s=50)
plt.xlabel('Temperature (°C)', fontsize=12)
plt.ylabel('Ice Cream Sales ($)', fontsize=12)
plt.title('Ice Cream Sales vs Temperature', fontsize=14)
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(temperature, sales, test_size=0.2, random_state=42)

# Compare models
models = {
    'Linear': Pipeline([('linear', LinearRegression())]),
    'Polynomial (degree 2)': Pipeline([
        ('poly', PolynomialFeatures(2)),
        ('linear', LinearRegression())
    ]),
    'Polynomial (degree 3)': Pipeline([
        ('poly', PolynomialFeatures(3)),
        ('linear', LinearRegression())
    ])
}

plt.figure(figsize=(15, 5))
X_plot = np.linspace(10, 40, 300).reshape(-1, 1)

for i, (name, model) in enumerate(models.items(), 1):
    model.fit(X_train, y_train)
    y_test_pred = model.predict(X_test)
    y_plot = model.predict(X_plot)
    
    plt.subplot(1, 3, i)
    plt.scatter(X_train, y_train, alpha=0.5, edgecolors='k', s=30, label='Training')
    plt.scatter(X_test, y_test, alpha=0.7, edgecolors='k', s=50, c='red', label='Test')
    plt.plot(X_plot, y_plot, 'b-', linewidth=2, label='Model')
    plt.xlabel('Temperature (°C)', fontsize=11)
    plt.ylabel('Sales ($)', fontsize=11)
    plt.title(f'{name}\nR²={r2_score(y_test, y_test_pred):.3f}', fontsize=12)
    plt.legend(fontsize=9)
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print detailed results
print("Model Performance Summary:")
print("=" * 60)
for name, model in models.items():
    model.fit(X_train, y_train)
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    print(f"\n{name}:")
    print(f"  Training R²: {r2_score(y_train, y_train_pred):.4f}")
    print(f"  Test R²: {r2_score(y_test, y_test_pred):.4f}")
    print(f"  Test RMSE: ${np.sqrt(mean_squared_error(y_test, y_test_pred)):.2f}")

## Key Takeaways

1. **Non-linear relationships** can be modeled using polynomial features while still using linear regression
2. **Feature explosion**: Polynomial features grow rapidly with degree and number of original features
   - For $n$ features and degree $d$: approximately $\binom{n+d}{d}$ features
3. **Overfitting risk**: Higher degree polynomials fit training data better but may generalize poorly
4. **Model selection**: Use cross-validation to choose the optimal polynomial degree
5. **Feature scaling** becomes even more important with polynomial features
6. **Interaction terms** (e.g., $x_1 \cdot x_2$) are automatically included

## When to Use Polynomial Regression

✅ **Use when:**
- Clear non-linear patterns in data
- Relatively small number of features
- Sufficient training data
- Physical relationship suggests polynomial form

❌ **Avoid when:**
- Many features (curse of dimensionality)
- Very high degrees needed (consider other models)
- Oscillatory or highly complex patterns
- Limited training data

## Next Steps

- **Regularization**: Ridge and Lasso to control polynomial complexity
- **Splines**: More flexible piecewise polynomial fits
- **Non-parametric methods**: Kernel regression, decision trees

## Exercises

1. Create a dataset with a cubic relationship and find the optimal polynomial degree
2. Explore how the number of features affects polynomial feature explosion
3. Implement cross-validation to automatically select the best polynomial degree
4. Compare polynomial regression with other non-linear methods (decision trees, neural networks)
5. Analyze the learned coefficients and their interpretability